In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

master = pd.read_parquet("hcp_analysis_clean.parquet")
eps = 1e-9

C = {
    "SEG_B":          "#1A6FD4",
    "SEG_C":          "#D4720A",
    "SEG_A":          "#6B7280",
    "No Clasificado": "#7C3AED",
    "positive":       "#0D9E6E",
    "negative":       "#DC3545",
    "muted":          "#6B7A96",
}

SEG_ORDER    = ["SEG_B", "SEG_C", "SEG_A"]
SEG_COLORS   = [C["SEG_B"], C["SEG_C"], C["SEG_A"]]
SEG_ALL      = ["SEG_B", "SEG_C", "SEG_A", "No Clasificado"]
SEG_ALL_COLS = [C["SEG_B"], C["SEG_C"], C["SEG_A"], C["No Clasificado"]]

# Map adoption stages to Spanish labels
ADOPTION_MAP = {
    "Never tried":    "Nunca prescribió",
    "Active":         "Activo",
    "Trialed — lapsed": "Inactivo",
}
master["ADOPTION_STAGE_ES"] = master["ADOPTION_STAGE"].map(ADOPTION_MAP).fillna(master["ADOPTION_STAGE"])

labeled   = master[master["ATSEG_LABEL"].notna()].copy()
unlabeled = master[master["ATSEG_LABEL"].isna()].copy()

def rank_norm(s):
    return s.rank(pct=True).fillna(0)

unlabeled = unlabeled.copy()
unlabeled["opportunity_score"] = (
    rank_norm(unlabeled["UC_TRX_mean"])             * 0.30 +
    rank_norm(unlabeled["BRAND1_TRX_mean"])         * 0.25 +
    rank_norm(unlabeled["BRAND1_TRX_trend_ratio"])  * 0.25 +
    rank_norm(unlabeled["new_patient_orientation"]) * 0.20
)
unlabeled["priority_tier"] = pd.cut(
    unlabeled["opportunity_score"],
    bins=[-0.01, 0.35, 0.60, 1.01],
    labels=["Nivel 3 — Monitorear", "Nivel 2 — Validar", "Nivel 1 — Inmediato"],
)
unlabeled["coverage"] = np.where(
    unlabeled["DETAILS_total"] == 0, "Sin visitas rep", "Cubierto"
)

print(f"Datos cargados: {len(master):,} HCPs × {master.shape[1]} columnas")
print(f"  Etiquetados: {len(labeled):,}  |  No etiquetados: {len(unlabeled):,}")


ModuleNotFoundError: No module named 'pandas'

## Distribución de Clases

In [ ]:
seg_counts = master["SEGMENT_DISPLAY"].value_counts()
labels = SEG_ALL
values = [int(seg_counts.get(s, 0)) for s in labels]
colors = SEG_ALL_COLS

fig = go.Figure(go.Pie(
    labels=labels,
    values=values,
    hole=0.52,
    marker=dict(colors=colors, line=dict(color="#FAFAF7", width=2)),
    textposition="inside",
    textinfo="label+percent",
    insidetextfont=dict(size=11, color="white"),
))
fig.update_layout(
    title="Distribución de Clases",
    template="plotly_white",
    annotations=[dict(
        text=f"{len(master):,}<br>HCPs",
        x=0.5, y=0.5,
        font=dict(size=14, color="#1A1F2E"),
        showarrow=False,
    )],
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
    width=600, height=480,
)
fig.show()


## Rasgos por Clase

In [ ]:
FEAT_MAP = {
    "UC_TRX_mean":                 "UC Recetas / sem",
    "BRAND1_TRX_mean":             "Pfizer Recetado / sem",
    "brand1_share_of_uc":          "Participación Pfizer en UC",
    "BRAND1_TRX_trend_ratio":      "Tendencia Pfizer",
    "BRAND1_TRX_is_growing":       "Pfizer en crecimiento (%)",
    "details_per_trx":             "Visitas rep / Rx",
    "established_therapy_loyalty": "Lealtad a terapia original",
    "new_patient_orientation":     "Orientación a nuevos pacientes",
}

seg_means = labeled.groupby("ATSEG_LABEL")[list(FEAT_MAP.keys())].mean()
seg_means = seg_means.loc[SEG_ORDER].rename(columns=FEAT_MAP)

seg_norm = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min() + 1e-9)

# z: features × segments
z         = seg_norm.T.values          # shape (8 feats, 3 segs)
text_raw  = seg_means.T.values         # raw values for annotation

feat_names = list(FEAT_MAP.values())
text_annot = [[f"{text_raw[fi, si]:.4f}" for si in range(len(SEG_ORDER))]
              for fi in range(len(feat_names))]

font_colors = [
    ["white" if z[fi, si] > 0.5 else "#1A1F2E"
     for si in range(len(SEG_ORDER))]
    for fi in range(len(feat_names))
]

fig = go.Figure(go.Heatmap(
    z=z,
    x=SEG_ORDER,
    y=feat_names,
    colorscale=[[0, "#F4F3EE"], [0.5, "#F5C97A"], [1, "#1A6FD4"]],
    zmin=0, zmax=1,
    text=text_annot,
    texttemplate="%{text}",
    textfont=dict(size=10),
    colorbar=dict(title="Normalizado", thickness=12),
    showscale=True,
))
fig.update_layout(
    title="Rasgos por Clase",
    template="plotly_white",
    xaxis=dict(title="Clase", side="bottom"),
    yaxis=dict(autorange="reversed"),
    width=680, height=420,
)
fig.show()


## Etapa de adopción Brand1

In [ ]:
STAGE_ES = ["Nunca prescribió", "Activo", "Inactivo"]
STAGE_COLORS = {
    "Nunca prescribió": "#DC3545",
    "Activo":           "#0D9E6E",
    "Inactivo":         "#D4720A",
}

adopt_pct = (
    pd.crosstab(labeled["ATSEG_LABEL"], labeled["ADOPTION_STAGE_ES"], normalize="index") * 100
).loc[SEG_ORDER, [s for s in STAGE_ES if s in labeled["ADOPTION_STAGE_ES"].unique()]]

fig = go.Figure()
for stage in adopt_pct.columns:
    vals = adopt_pct[stage].tolist()
    text = [f"{v:.1f}%" if v > 5 else "" for v in vals]
    fig.add_trace(go.Bar(
        name=stage,
        x=SEG_ORDER,
        y=vals,
        marker_color=STAGE_COLORS.get(stage, C["muted"]),
        text=text,
        textposition="inside",
        insidetextfont=dict(color="white", size=10),
    ))

fig.update_layout(
    barmode="stack",
    title="Etapa de adopción Brand1",
    template="plotly_white",
    yaxis=dict(title="% del segmento", range=[0, 105]),
    xaxis=dict(title="Clase"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5),
    width=500, height=440,
)
fig.show()


## Crecimiento por Segmento

In [ ]:
METRICS = {
    "UC Recetas / sem":          "UC_TRX_mean",
    "Participación Pfizer en UC":"brand1_share_of_uc",
    "Pfizer en crecimiento (%)": "BRAND1_TRX_is_growing",
    "Visitas rep / Rx":          "details_per_trx",
}

SEG_PLOT_ORDER = ["SEG_A", "SEG_B", "SEG_C"]
SEG_PLOT_COLS  = [C["SEG_A"], C["SEG_B"], C["SEG_C"]]

fig = make_subplots(rows=1, cols=4, subplot_titles=list(METRICS.keys()))

for col_idx, (metric_name, col_name) in enumerate(METRICS.items(), start=1):
    vals = [labeled[labeled["ATSEG_LABEL"] == s][col_name].mean() for s in SEG_PLOT_ORDER]
    text = [f"{v:.4f}" for v in vals]
    fig.add_trace(
        go.Bar(
            x=SEG_PLOT_ORDER,
            y=vals,
            marker_color=SEG_PLOT_COLS,
            text=text,
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title="Crecimiento por Segmento",
    template="plotly_white",
    width=950, height=420,
)
fig.show()


## Recetas Totales UC vs Recetas Pfizer

In [ ]:
d = labeled.sample(min(3000, len(labeled)), random_state=42)
d["Clase"] = d["ATSEG_LABEL"]

fig = px.scatter(
    d,
    x="UC_TRX_mean",
    y="brand1_share_of_uc",
    color="Clase",
    color_discrete_map={"SEG_B": C["SEG_B"], "SEG_C": C["SEG_C"], "SEG_A": C["SEG_A"]},
    category_orders={"Clase": ["SEG_A", "SEG_C", "SEG_B"]},
    labels={
        "UC_TRX_mean":        "Recetas UC semanal promedio",
        "brand1_share_of_uc": "Participación Pfizer en recetas UC",
        "Clase":              "Clase",
    },
    title="Recetas Totales UC vs Recetas Pfizer",
    opacity=0.4,
)
fig.update_traces(marker=dict(size=5, line=dict(width=0)))
fig.update_layout(template="plotly_white", width=680, height=480)
fig.show()


## Tipo de Medicamento por Clase

In [ ]:
MIX = {
    "UC_TRX_mean":   "TRX UC total",
    "IL23_TRX_mean": "TRX IL-23 (protocolo)",
    "ORAL_TRX_mean": "TRX Oral (pref. paciente)",
}
MIX_COLORS = [C["muted"], C["SEG_C"], C["SEG_B"]]

mix_means = labeled.groupby("ATSEG_LABEL")[list(MIX.keys())].mean()
mix_means = mix_means.loc[SEG_PLOT_ORDER].rename(columns=MIX)

fig = go.Figure()
for (col_name, bar_color) in zip(mix_means.columns, MIX_COLORS):
    fig.add_trace(go.Bar(
        name=col_name,
        x=SEG_PLOT_ORDER,
        y=mix_means[col_name].tolist(),
        marker_color=bar_color,
    ))

fig.update_layout(
    barmode="group",
    title="Tipo de Medicamento por Clase",
    template="plotly_white",
    xaxis=dict(title="Clase"),
    yaxis=dict(title="Recetas semanales promedio"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5),
    width=620, height=460,
)
fig.show()


## Adopcion Pfizer

In [ ]:
rows_list = []
for seg in SEG_ORDER:
    s = labeled[labeled["ATSEG_LABEL"] == seg]
    tot = len(s)
    for stage_es in STAGE_ES:
        cnt = (s["ADOPTION_STAGE_ES"] == stage_es).sum()
        rows_list.append({"Segmento": seg, "Etapa": stage_es, "Count": int(cnt), "Pct": cnt / tot * 100})

df_funnel = pd.DataFrame(rows_list)

fig = go.Figure()
for stage_es in STAGE_ES:
    d_stage = df_funnel[df_funnel["Etapa"] == stage_es]
    text = [f"{int(v):,}" for v in d_stage["Count"]]
    fig.add_trace(go.Bar(
        name=stage_es,
        x=d_stage["Segmento"].tolist(),
        y=d_stage["Count"].tolist(),
        marker_color=STAGE_COLORS.get(stage_es, C["muted"]),
        text=text,
        textposition="outside",
        textfont=dict(size=9),
    ))

fig.update_layout(
    barmode="group",
    title="Adopcion Pfizer",
    template="plotly_white",
    xaxis=dict(title="Clase"),
    yaxis=dict(title="Numero de HCPs"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5),
    width=700, height=470,
)
fig.show()


## Tendencia Pfizer Semanal

In [ ]:
# Approximate weekly trend using average vs recent-8-week mean per segment.
# T1 mean ≈ BRAND1_TRX_mean - BRAND1_TRX_trend_delta/2  (early tercile proxy)
# T3 mean = T1 mean + BRAND1_TRX_trend_delta  (late tercile proxy)

trend_df = (
    labeled.groupby("ATSEG_LABEL")[["BRAND1_TRX_mean", "BRAND1_TRX_recent8_mean"]]
    .mean()
    .loc[SEG_ORDER]
    .reset_index()
    .rename(columns={"ATSEG_LABEL": "Clase"})
)

fig = go.Figure()
for seg, color in zip(SEG_ORDER, SEG_COLORS):
    row = trend_df[trend_df["Clase"] == seg].iloc[0]
    fig.add_trace(go.Bar(
        name=f"{seg} – Promedio 86 sem",
        x=[f"{seg} (Promedio)"],
        y=[row["BRAND1_TRX_mean"]],
        marker_color=color,
        opacity=0.6,
        legendgroup=seg,
    ))
    fig.add_trace(go.Bar(
        name=f"{seg} – Últimas 8 sem",
        x=[f"{seg} (Reciente)"],
        y=[row["BRAND1_TRX_recent8_mean"]],
        marker_color=color,
        opacity=1.0,
        legendgroup=seg,
    ))

fig.update_layout(
    barmode="group",
    title="Tendencia Pfizer Semanal",
    template="plotly_white",
    xaxis=dict(title="Segmento / Período"),
    yaxis=dict(title="Brand1 TRX prom / HCP / sem"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.4, xanchor="center", x=0.5),
    width=750, height=480,
)
fig.show()


## Señales de Crecimiento de Pfizer

In [ ]:
signals_map = {
    "BRAND1_TRX_is_growing":     "B1 En crecimiento (%)",
    "BRAND1_TRX_is_new_adopter": "Nuevo adoptante (%)",
}
SIGNAL_LABELS = list(signals_map.values()) + ["Activo últimas 8 sem (%)"]

sig_rows = []
for seg in SEG_ORDER:
    s = labeled[labeled["ATSEG_LABEL"] == seg]
    for col, name in signals_map.items():
        sig_rows.append({"Segmento": seg, "Señal": name, "Pct": s[col].mean() * 100})
    sig_rows.append({
        "Segmento": seg,
        "Señal": "Activo últimas 8 sem (%)",
        "Pct": (s["BRAND1_TRX_recent8_mean"] > 0).mean() * 100,
    })

df_sig = pd.DataFrame(sig_rows)

fig = go.Figure()
for seg, color in zip(SEG_ORDER, SEG_COLORS):
    d_seg = df_sig[df_sig["Segmento"] == seg]
    vals  = d_seg.set_index("Señal")["Pct"].reindex(SIGNAL_LABELS).tolist()
    text  = [f"{v:.2f}%" for v in vals]
    fig.add_trace(go.Bar(
        name=seg,
        x=SIGNAL_LABELS,
        y=vals,
        marker_color=color,
        text=text,
        textposition="outside",
        textfont=dict(size=8),
    ))

fig.update_layout(
    barmode="group",
    title="Señales de Crecimiento de Pfizer",
    template="plotly_white",
    xaxis=dict(title=""),
    yaxis=dict(title="% de HCPs en el segmento"),
    legend=dict(title="Clase"),
    width=750, height=470,
)
fig.show()


## Distribución del Ratio de Tendencia Brand1

In [ ]:
fig = go.Figure()
for seg, color in zip(["SEG_A", "SEG_C", "SEG_B"], [C["SEG_A"], C["SEG_C"], C["SEG_B"]]):
    vals = labeled[labeled["ATSEG_LABEL"] == seg]["BRAND1_TRX_trend_ratio"].clip(0, 3)
    fig.add_trace(go.Histogram(
        x=vals,
        name=seg,
        marker_color=color,
        opacity=0.65,
        nbinsx=40,
        xbins=dict(start=0, end=3, size=3/40),
    ))

fig.add_vline(x=1.0, line_dash="dash", line_color="#0D9E6E", line_width=1.5,
              annotation_text="1.0 = estable", annotation_position="top right")
fig.add_vline(x=1.1, line_dash="dot",  line_color="#1A6FD4", line_width=1.5,
              annotation_text=">1.1 = creciendo", annotation_position="top left")

fig.update_layout(
    barmode="overlay",
    title="Distribución del Ratio de Tendencia Brand1",
    template="plotly_white",
    xaxis=dict(title="Ratio de tendencia (T3/T1)"),
    yaxis=dict(title="Numero de HCPs"),
    legend=dict(title="Clase"),
    width=680, height=460,
)
fig.show()


## Engagement de Representante por Clase

In [ ]:
ENG_PANELS = [
    ("details_per_trx",    "Visitas rep / Pfizer Rx",  "Visitas por Rx"),
    ("promo_channel_count","Canales promo utilizados",  "Canales promedio"),
    ("DETAILS_total",      "Visitas totales",           "Visitas totales"),
]

SEG_ENG_ORDER = ["SEG_A", "SEG_B", "SEG_C"]
SEG_ENG_COLS  = [C["SEG_A"], C["SEG_B"], C["SEG_C"]]

fig = make_subplots(rows=1, cols=3, subplot_titles=[p[1] for p in ENG_PANELS])

for col_idx, (col_name, panel_title, y_label) in enumerate(ENG_PANELS, start=1):
    vals = [labeled[labeled["ATSEG_LABEL"] == s][col_name].mean() for s in SEG_ENG_ORDER]
    text = [f"{v:.2f}" for v in vals]
    fig.add_trace(
        go.Bar(
            x=SEG_ENG_ORDER,
            y=vals,
            marker_color=SEG_ENG_COLS,
            text=text,
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=col_idx,
    )
    fig.update_yaxes(title_text=y_label, row=1, col=col_idx)

fig.update_layout(
    title="Engagement de Representante por Clase",
    template="plotly_white",
    width=900, height=430,
)
fig.show()


## Visitas Rep vs Prescripciones Pfizer

In [ ]:
d2 = labeled.sample(min(2500, len(labeled)), random_state=42).copy()
d2["Clase"] = d2["ATSEG_LABEL"]

fig = px.scatter(
    d2,
    x="DETAILS_total",
    y="BRAND1_TRX_mean",
    color="Clase",
    color_discrete_map={"SEG_A": C["SEG_A"], "SEG_C": C["SEG_C"], "SEG_B": C["SEG_B"]},
    category_orders={"Clase": ["SEG_A", "SEG_C", "SEG_B"]},
    labels={
        "DETAILS_total":    "Visitas totales representantes",
        "BRAND1_TRX_mean":  "Pfizer recetas semanales promedio",
        "Clase":            "Clase",
    },
    title="Visitas Rep vs Prescripciones Pfizer",
    opacity=0.4,
)
fig.update_traces(marker=dict(size=5, line=dict(width=0)))
fig.update_layout(template="plotly_white", width=680, height=500)
fig.show()


## Pfizer vs Brand2 en Mercado por Clase

In [ ]:
comp_map = {
    "brand1_share_of_uc":   "Pfizer en UC (%)",
    "comp_brand2_share_uc": "Brand2 en UC (%)",
}
comp_colors = [C["SEG_B"], C["SEG_C"]]

comp_means = labeled.groupby("ATSEG_LABEL")[list(comp_map.keys())].mean() * 100
comp_means = comp_means.loc[SEG_PLOT_ORDER].rename(columns=comp_map)

fig = go.Figure()
for (metric, col_color) in zip(comp_means.columns, comp_colors):
    vals = comp_means[metric].tolist()
    text = [f"{v:.3f}%" for v in vals]
    fig.add_trace(go.Bar(
        name=metric,
        x=SEG_PLOT_ORDER,
        y=vals,
        marker_color=col_color,
        text=text,
        textposition="outside",
        textfont=dict(size=9),
    ))

fig.update_layout(
    barmode="group",
    title="Pfizer vs Brand2 en Mercado por Clase",
    template="plotly_white",
    xaxis=dict(title="Clase"),
    yaxis=dict(title="% del TRX UC total"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    width=620, height=460,
)
fig.show()


## Brand2 / Pfizer por Clase

In [ ]:
ratios = []
for seg in SEG_PLOT_ORDER:
    s  = labeled[labeled["ATSEG_LABEL"] == seg]
    b2 = s["BRAND2_TRX_mean"].mean()
    b1 = s["BRAND1_TRX_mean"].mean()
    ratios.append(b2 / (b1 + eps))

text = [f"{v:.2f}×" for v in ratios]

fig = go.Figure(go.Bar(
    x=SEG_PLOT_ORDER,
    y=ratios,
    marker_color=SEG_PLOT_COLS,
    text=text,
    textposition="outside",
    textfont=dict(size=13, color="#1A1F2E"),
))
fig.update_layout(
    title="Brand2 / Pfizer por Clase",
    template="plotly_white",
    xaxis=dict(title="Clase"),
    yaxis=dict(title="Brand2 / Pfizer"),
    showlegend=False,
    width=500, height=440,
)
fig.show()


## Distribución de Especialidades por Clase

In [ ]:
ct = pd.crosstab(labeled["SPECIALTY"], labeled["ATSEG_LABEL"], normalize="index") * 100
ct = ct.reindex(columns=["SEG_A", "SEG_B", "SEG_C"]).fillna(0)

z_vals    = ct.values                 # shape: specialties × segments
feat_list = ct.index.tolist()
seg_list  = ct.columns.tolist()
text_annot = [[f"{ct.loc[sp, sg]:.0f}%" for sg in seg_list] for sp in feat_list]

fig = go.Figure(go.Heatmap(
    z=z_vals,
    x=seg_list,
    y=feat_list,
    colorscale=[[0, "#F4F3EE"], [0.5, "#F5C97A"], [1, "#1A6FD4"]],
    text=text_annot,
    texttemplate="%{text}",
    textfont=dict(size=11),
    colorbar=dict(title="% de especialidad", thickness=12),
    showscale=True,
))
fig.update_layout(
    title="Distribución de Especialidades por Clase",
    template="plotly_white",
    xaxis=dict(title="Clase"),
    yaxis=dict(autorange="reversed"),
    width=600, height=400,
)
fig.show()


## HCPs por Especialidad y Segmento

In [ ]:
spec_seg = pd.crosstab(labeled["SPECIALTY"], labeled["ATSEG_LABEL"])
spec_seg = spec_seg.reindex(columns=["SEG_A", "SEG_B", "SEG_C"]).fillna(0)
spec_seg = spec_seg.sort_values("SEG_B", ascending=True)

fig = go.Figure()
for seg, color in [("SEG_A", C["SEG_A"]), ("SEG_B", C["SEG_B"]), ("SEG_C", C["SEG_C"])]:
    if seg in spec_seg.columns:
        fig.add_trace(go.Bar(
            name=seg,
            y=spec_seg.index.tolist(),
            x=spec_seg[seg].tolist(),
            orientation="h",
            marker_color=color,
        ))

fig.update_layout(
    barmode="stack",
    title="HCPs por Especialidad y Segmento",
    template="plotly_white",
    xaxis=dict(title="Numero de HCPs"),
    yaxis=dict(title="Especialidad"),
    legend=dict(title="Clase", orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    width=680, height=400,
)
fig.show()


## Distribución de P de Recetar

In [ ]:
fig = go.Figure(go.Histogram(
    x=unlabeled["opportunity_score"],
    nbinsx=50,
    marker_color=C["No Clasificado"],
    opacity=0.80,
    name="No Clasificado",
))
fig.update_layout(
    title="Distribución de P de Recetar",
    template="plotly_white",
    xaxis=dict(title="Probabilidad de Recetar"),
    yaxis=dict(title="Numero de HCPs"),
    showlegend=False,
    width=640, height=450,
)
fig.show()


## P de Recetar vs Visitas Rep

In [ ]:
high_opp = unlabeled[unlabeled["opportunity_score"] >= 0.35].sample(
    min(1500, (unlabeled["opportunity_score"] >= 0.35).sum()), random_state=42
).copy()

COV_COLORS = {"Sin visitas rep": "#DC3545", "Cubierto": "#1A6FD4"}
COV_SYMBOLS = {"Sin visitas rep": "circle", "Cubierto": "square"}

fig = go.Figure()
for cov, color in COV_COLORS.items():
    s = high_opp[high_opp["coverage"] == cov]
    fig.add_trace(go.Scatter(
        x=s["UC_TRX_mean"],
        y=s["opportunity_score"],
        mode="markers",
        name=cov,
        marker=dict(
            color=color,
            size=6,
            opacity=0.6,
            symbol=COV_SYMBOLS[cov],
            line=dict(width=0),
        ),
    ))

fig.update_layout(
    title="P de Recetar vs Visitas Rep",
    template="plotly_white",
    xaxis=dict(title="Recetas UC semanal promedio"),
    yaxis=dict(title="Oportunidad"),
    legend=dict(title="Cobertura"),
    width=680, height=490,
)
fig.show()
